# 05 — Train model E2: frozen EfficientNet-B0

**Goal:** train the second model of the experiment plan and save it so notebooks 03 and 04 can score and inspect it.

**E2 in one sentence:** take EfficientNet-B0, a network that has already learned to recognise everyday objects from
1.2 million ImageNet photos, keep everything it learned **frozen**, and only train a small new last layer that answers
"real or fake?".

**Why this experiment?** E1 trained a small network from zero. E2 asks: *are features learned on ordinary photos
already good enough to spot AI-generated faces?* It is fast to train because only the last layer learns.

**Labels:** `fake` = 1 (what we want to catch), `real` = 0. The model outputs `P(fake)`, a number from 0 to 1.

**How to use:** run the cells from top to bottom (Shift+Enter). Works on your own computer and in Google Colab.
**Use a GPU on Colab:** Runtime → Change runtime type → T4 GPU. Training only starts when you switch it on (Step 7).

## The model at a glance

```
face photo (224 x 224 pixels, 3 colours)
        │
        ▼
EfficientNet-B0 "feature extractor"     ← pretrained on ImageNet, FROZEN (does not change)
        │   turns the photo into 1,280 numbers that describe it
        ▼
dropout 20%                              ← randomly ignores some numbers while training, to avoid memorising
        │
        ▼
new last layer: 1,280 numbers → 1 number ← the ONLY part that is trained in E2
        │
        ▼
sigmoid → P(fake) between 0 and 1        ← 0.5 or more = "fake"
```

The other experiments change one thing at a time: E3 also trains the frozen part (fine-tuning), E4 unfreezes it
gradually, E5 uses ResNet-50 instead, and E6 is the control trained on clean photos only.

## Step 1 — Find the project and get the tools ready

**What this does, in plain words** (same start as notebooks 01–04):

1. Finds the project folder. On Colab it downloads (clones) it from GitHub first.
2. Moves into the project folder, because the settings use paths like `data/raw/...` and `outputs/` that start there.
3. Installs any missing package the training code needs (Colab already has PyTorch, so it is never reinstalled).
4. Tells you whether a GPU is available.

In [ ]:
# 1. Tools we need
import importlib.util
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image

# 2. Find the project folder: look in the current folder and every folder above it.
PROJECT_ROOT = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src" / "deepfake_kyc").is_dir():
        PROJECT_ROOT = folder
        break

# Not found (for example on Colab)? Download the project from GitHub into the current folder.
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd() / "UOT_SCS_3546_041_Deep-Learning"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git",
                        str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)                           # settings paths start from the project folder
sys.path.insert(0, str(PROJECT_ROOT / "src"))    # lets Python import the project's own code
print("Project folder:", PROJECT_ROOT)

# 3. Install missing packages: {name used in "import": name used by pip}
needed = {"timm": "timm", "sklearn": "scikit-learn", "yaml": "PyYAML", "tqdm": "tqdm"}
for module, package in needed.items():
    if importlib.util.find_spec(module) is None:
        print("Installing", package, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# 4. GPU or not?
import torch

print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: training will be slow. On Colab choose Runtime > Change runtime type > T4 GPU, then run again.")

## Step 2 — Load the E2 settings

**What this does:** reads `configs/experiments/02_effnet_frozen.yaml`. That file only lists what E2 changes;
everything else comes from `configs/base.yaml`, shared by all experiments. The cell prints the important settings
in plain words so you know exactly what will be trained.

In [ ]:
from deepfake_kyc.config import load_config

CONFIG_FILE = PROJECT_ROOT / "configs" / "experiments" / "02_effnet_frozen.yaml"
cfg = load_config(CONFIG_FILE)

frozen = cfg["train"]["freeze_backbone_epochs"] >= cfg["train"]["epochs"]
print("Experiment:     ", cfg["experiment_name"])
print("Model:          ", cfg["model"]["name"], "(pretrained on ImageNet)" if cfg["model"]["pretrained"] else "")
print("Pretrained part:", "frozen for the whole run" if frozen else "also trained")
print("Image size:     ", cfg["data"]["image_size"], "x", cfg["data"]["image_size"])
print("Epochs:         ", cfg["train"]["epochs"], "(passes over the training photos)")
print("Batch size:     ", cfg["data"]["batch_size"], "photos at a time")
print("Learning rate:  ", cfg["train"]["lr"])
print("Trained on:     ", "real + fake faces, half randomly damaged (blur, JPEG, ...)"
      if cfg["augment"]["degradation"] else "real + fake faces, clean only")
print("Random seed:    ", cfg["seed"], "(same seed = same result every run)")

## Step 3 — Make sure the dataset is here

**What this does:** counts the images in `train`, `valid` and `test`. If they are missing, it can get them for you.
Pick **one** way with `DATA_SOURCE`:

| `DATA_SOURCE` | What happens | When to use |
|---|---|---|
| `"drive"` | copies `archive.zip` from your Google Drive and unpacks it (same file as notebook 01, Step 2a Drive) | Colab, zip already in Drive |
| `"kaggle"` | downloads the zip straight from Kaggle (fastest on Colab, 1–3 min) | Colab, no zip in Drive |
| `"none"` | does nothing | your own computer: use notebook 01, Step 2 |

For `"kaggle"` you need a free Kaggle account: accept the dataset's rules on its
[Kaggle page](https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces), then create an API key under
kaggle.com → Settings → API. The cell asks for your username and key; the key is hidden and never saved in the notebook.

On Colab each notebook runs on its own fresh computer, so the data from notebook 01 is not here; run this step.

In [ ]:
DATA_SOURCE = "drive" if "google.colab" in sys.modules else "none"   # "drive", "kaggle" or "none"
DRIVE_ZIP = "/content/drive/MyDrive/archive.zip"                      # same as notebook 01

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_ROOT = PROJECT_ROOT / cfg["data"]["root"]       # folder with train/ valid/ test/
SPLITS = ["train", "valid", "test"]

def list_images(split, class_name):
    """All .jpg files in one folder, e.g. list_images("train", "fake")."""
    return sorted((DATA_ROOT / split / class_name).glob("*.jpg"))

def data_ready():
    """True if every split has real and fake images."""
    return all(list_images(split, "real") and list_images(split, "fake") for split in SPLITS)

if not data_ready() and DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    print("Unpacking", DRIVE_ZIP, "... (a few minutes)")
    with zipfile.ZipFile(DRIVE_ZIP) as archive:
        archive.extractall(RAW_DATA_DIR)

if not data_ready() and DATA_SOURCE == "kaggle":
    from getpass import getpass
    os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
    os.environ["KAGGLE_KEY"] = getpass("Kaggle API key (hidden): ")
    if importlib.util.find_spec("kaggle") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    print("Downloading about 4 GB from Kaggle ...")
    subprocess.run([sys.executable, "-m", "kaggle", "datasets", "download", "-d",
                    "xhlulu/140k-real-and-fake-faces", "-p", str(RAW_DATA_DIR), "--unzip"], check=True)

for split in SPLITS:
    print(f"{split:6} real={len(list_images(split, 'real')):,}  fake={len(list_images(split, 'fake')):,}")
DATA_AVAILABLE = data_ready()
print("Dataset ready." if DATA_AVAILABLE else "Dataset missing: choose a DATA_SOURCE above, or use notebook 01, Step 2.")

## Step 4 — Where to save the trained model

**What this does:** chooses the folder for the results.

- **Colab:** your Google Drive, `MyDrive/deepfake_kyc_outputs/`, because everything else on Colab is deleted when
  the session ends. Notebooks 03 and 04 look in this same Drive folder.
- **Your own computer:** the project's `outputs/` folder, like `train.py`.

The model will be saved in a sub-folder named after the experiment, e.g. `.../02_effnet_frozen/`.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")      # asks you to allow access to your Drive
    OUTPUT_DIR = Path("/content/drive/MyDrive/deepfake_kyc_outputs")
else:
    OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results will be saved in:", OUTPUT_DIR)

## Step 5 — A quick look at the training photos

**What this does:** shows 3 real and 3 fake training faces. A quick check that the images and labels look right
(notebook 01 has the full data exploration).

In [ ]:
import random

if DATA_AVAILABLE:
    random.seed(cfg["seed"])
    examples = [(path, "real (0)") for path in random.sample(list_images("train", "real"), 3)] + \
               [(path, "fake (1)") for path in random.sample(list_images("train", "fake"), 3)]

    fig, axes = plt.subplots(1, 6, figsize=(15, 3))
    for ax, (path, label) in zip(axes, examples):
        with Image.open(path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(label)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: dataset not available (Step 3).")

## Step 6 — Build the model and check it

**What this does:**

1. Builds EfficientNet-B0 with ImageNet weights and a new 1-output last layer
   (the first time, `timm` downloads about 20 MB of weights).
2. Freezes the pretrained part, exactly like the training code does for E2.
3. Counts how many numbers (parameters) the model has and how many will actually learn.
4. Pushes 2 blank test images through it to check the output shape: one number per photo.
   This only checks the wiring; it is not a result.

In [ ]:
from deepfake_kyc.models import build_model, set_backbone_trainable, split_parameters
from deepfake_kyc.utils import get_device, seed_everything

seed_everything(cfg["seed"])     # same random numbers every run
device = get_device()            # GPU if there is one
print("Running on:", device)

model = build_model(cfg["model"]["name"], pretrained=cfg["model"]["pretrained"],
                    dropout=cfg["model"]["dropout"]).to(device)
set_backbone_trainable(model, trainable=False)          # freeze the pretrained part

pretrained_part, last_layer = split_parameters(model)
count = lambda params: sum(p.numel() for p in params)
print(f"All parameters:            {count(model.parameters()):,}")
print(f"Pretrained part (frozen):  {count(pretrained_part):,}")
print(f"New last layer (learns):   {count(last_layer):,}")

size = cfg["data"]["image_size"]
blank_photos = torch.zeros(2, 3, size, size, device=device)    # 2 photos, 3 colours, size x size
model.eval()
with torch.no_grad():
    output = model(blank_photos)
print("Output shape:", tuple(output.shape), "-> one number per photo (expected (2, 1))")

## Step 7 — Train (switch on when ready)

**What happens during training:**

1. The model sees a batch of training photos (half of them randomly damaged, see Step 2) and guesses `P(fake)` for each.
2. The **loss** (binary cross-entropy) measures how wrong the guesses were.
3. The optimiser (AdamW) nudges the last layer's numbers to make the loss smaller. Repeat for every batch = one **epoch**.
4. After each epoch the model is scored on the **validation** photos (never damaged, never used for learning).
5. The epoch with the lowest validation loss is saved as `best.pt`. Training stops early if validation stops improving.

**Controls:**
- `RUN_TRAINING = True` starts it.
- `QUICK_RUN = True`: a short test run (1,000 training photos, 300 validation photos, 1 epoch, a few minutes).
  It is saved as `02_effnet_frozen_notebook_demo`, so it never mixes with a real run
  (notebooks 03 and 04 will still list it; just ignore it when comparing experiments).
- `QUICK_RUN = False`: the real E2 run with all 100,000 training photos and the epochs from the settings,
  saved as `02_effnet_frozen` (the same name `train.py` uses). On a T4 GPU expect roughly half an hour or more.

The test photos are **not** used here; they are kept for the final score in Step 9.

In [ ]:
RUN_TRAINING = False     # change to True to train
QUICK_RUN = True         # True = short test run; False = the full E2 run

if RUN_TRAINING and not DATA_AVAILABLE:
    print("Cannot train: the dataset is missing (Step 3).")
elif RUN_TRAINING:
    from deepfake_kyc.train import main as run_training

    # Changes on top of the settings file, written as "setting=value" like in a terminal.
    changes = [f"output_dir={OUTPUT_DIR.as_posix()}", "data.num_workers=0"]
    if QUICK_RUN:
        changes += ["data.train_subset=1000", "data.val_subset=300", "train.epochs=1",
                    f"experiment_name={cfg['experiment_name']}_notebook_demo"]

    summary = run_training(["--config", str(CONFIG_FILE), "--set", *changes])
    RUN_DIR = OUTPUT_DIR / summary["experiment_name"]
    print("Saved in:", RUN_DIR)
else:
    print("Skipped. Change RUN_TRAINING to True to train.")

## Step 8 — Read the training results

**What this does:** opens the files the training saved and shows them:

| File | What is inside |
|---|---|
| `best.pt` | the trained model (used by notebooks 03 and 04) |
| `config.yaml` | the exact settings of this run |
| `history.csv` | one row per epoch: training loss, validation loss and validation scores |
| `summary.json` | which epoch was best and its scores |

**How to read the chart:** both losses should go down. If the training loss keeps falling but the validation loss
goes up, the model is starting to memorise the training photos (overfitting).

These are **validation** scores. Use them to choose settings; the test score comes last (Step 9).

In [ ]:
# Use the run from Step 7; if you trained earlier, pick the newest run of this experiment that exists.
if "RUN_DIR" not in globals():
    names = [cfg["experiment_name"], cfg["experiment_name"] + "_notebook_demo"]
    runs = [OUTPUT_DIR / name for name in names if (OUTPUT_DIR / name / "summary.json").exists()]
    RUN_DIR = max(runs, key=lambda folder: (folder / "summary.json").stat().st_mtime) if runs else None

if RUN_DIR is not None and (RUN_DIR / "summary.json").exists():
    summary = json.loads((RUN_DIR / "summary.json").read_text())
    history = pd.read_csv(RUN_DIR / "history.csv")
    print("Run:", RUN_DIR.name, "| best epoch:", summary["best_epoch"])
    display(history.round(4))

    plt.figure(figsize=(6, 3.5))
    plt.plot(history["epoch"], history["train_loss"], marker="o", label="training loss")
    plt.plot(history["epoch"], history["val_loss"], marker="o", label="validation loss")
    plt.xlabel("epoch")
    plt.ylabel("loss (lower is better)")
    plt.title(f"{RUN_DIR.name}: loss per epoch")
    plt.legend()
    plt.show()
else:
    print("No finished run found yet in", OUTPUT_DIR, "-> train in Step 7 first.")

## Step 9 (once, at the very end) — Score the model on the test photos

**Best practice:** look at the test photos **only once**, after you have finished choosing settings with the
validation scores. If you change settings because of the test score, the test score is no longer honest.

**What this does:** runs `evaluate.py` on the test photos and saves `eval_test.csv` next to `best.pt`,
exactly like notebook 03, Step 1b, so you can use either one. It tests the model the way it was trained:
a model trained on clean + damaged photos is also tested on damaged photos (26 tests, slower).

If `eval_test.csv` already exists (for example from notebook 03), this cell does not score again.
Notebook 03 then shows the results.

In [ ]:
RUN_TEST = False         # change to True for the final test score
TEST_PHOTOS = None       # None = all 20,000 test photos (the real final score); e.g. 2000 = quicker

if RUN_TEST and (RUN_DIR is None or not (RUN_DIR / "best.pt").exists()):
    print("No trained model found: train in Step 7 first.")
elif RUN_TEST and (RUN_DIR / "eval_test.csv").exists():
    print("Already scored:", RUN_DIR / "eval_test.csv", "-> see the results in notebook 03.")
elif RUN_TEST:
    from deepfake_kyc.evaluate import main as run_evaluate

    arguments = ["--checkpoint", str(RUN_DIR / "best.pt"), "--set", "data.num_workers=0"]
    if TEST_PHOTOS is not None:
        arguments.append(f"data.test_subset={TEST_PHOTOS}")
    if cfg["augment"]["degradation"]:              # trained with damage -> also test with damage
        arguments += ["--degradation", "all"]
    rows = run_evaluate(arguments)
    display(pd.DataFrame(rows).round(3))
else:
    print("Skipped. Change RUN_TEST to True only when you have finished choosing settings.")

## What comes next?

- **Compare:** run [03_results_charts.ipynb](03_results_charts.ipynb) to compare E2 with the other experiments,
  and [04_error_analysis.ipynb](04_error_analysis.ipynb) to see which photos it gets wrong.
- **Next experiment:** E3 (`configs/experiments/03_effnet_finetune.yaml`) also trains the pretrained part, with a
  smaller learning rate. To train it here, change `CONFIG_FILE` in Step 2 and run the notebook again.
- Use the same seed and the same data for every experiment, so the comparison is fair.

**Limits and responsible use:** this research model tells apart the kind of photos in this dataset. It does not
check who a person is and does not prove fraud. Results may not hold for other AI generators, cameras or groups of
people. Never use it as the only basis for a real decision, and never use private customer photos.